# RMUC 2026 哨兵行为树插件文档

本文档介绍 `rmuc_2026` 行为树系统中所有自定义插件的功能、端口和使用方法。  
行为树主入口：`rmuc_2026.xml`（BT.CPP v4 格式），共引用 **12** 棵子树（`DeathAndRespawn.xml` 存在但未被引用，属于孤立文件），涉及 **57** 个自定义插件。

> ⚠️ 消息类型已从单一 `RMUC.msg` 拆分为 **8 个独立小消息**，各自拥有独立话题。  
> 详见 `RMUC_Msg_Split_Doc.ipynb`。

---

## 1. 架构总览

```
rmuc_2026 (主树, ReactiveSequence 容器)
  ├── PerceptionAndBlackboard   → 12 话题订阅 + 黑板解析
  ├── InitOnce                  → 配置初始化
  ├── WhileDoElse (比赛阶段判断)
  │   ├── CommandHub            → 指令决策 + 裁判发送 (5Hz)
  │   └── ReactiveFallback (优先级从高到低, SetBlackboard active_subtree 跟踪)
  │       ├── [0] RespawnRecovery   → 死亡/复活/刷卡回血
  │       ├── [1] CriticalSurvival  → 危急生存
  │       ├── [2] BaseDefense       → 基地防御
  │       ├── [3] EngageCombat      → 交战
  │       ├── [4] SustainAndEconomy → 后勤补给
  │       ├── [5] ObjectivePlanner  → 目标占领
  │       └── [6] PatrolAndScan     → 巡逻扫描
  └── 非比赛阶段 → Home + 停火
```

## 2. 插件分类总览（57 个）

| 分类 | 代号 | 数量 | 说明 |
|:---|:---:|:---:|:---|
| **订阅插件** | A | 12 | ROS 话题订阅，写入黑板 |
| **初始化/配置插件** | B | 2 | 一次性配置参数写入 |
| **解析插件** | C | 1 | 集中黑板解析（7原始+7新→~70+输出） |
| **决策插件** | D | 4 | 姿态/经济/复活决策 |
| **命令复用插件** | E | 1 | 裁判系统命令发送 5Hz→0x0120 |
| **导航插件** | F | 4 | 导航目标发送/取消/移动 |
| **战斗插件** | G | 3 | 目标选择/瞄准/开火 |
| **机器人控制插件** | H | 1 | 底盘/云台/开火控制 |
| **导航选择插件** | I | 8 | 目标/补给/撤退/回血/补弹选择 |
| **恢复插件** (NEW) | J | 4 | 导航控制/等待回血/搜索计时/微搜索 |
| **条件插件** | K | 16 | 状态/阈值/区域检测 |
| **装饰器** | L | 1 | 频率控制 |
| **合计** | — | **57** | — |

## 3. A. 订阅插件（12 个）

所有订阅插件均为 RosTopicSubNode (Action)，负责将 ROS 话题数据写入黑板。

| # | BT 节点名 | 话题 | 输出端口 | 备注 |
|:---:|:---|:---|:---|:---|
| 1 | `RmucSubGameStatus` | `/game_status` | `game_status`, `now_ms` | — |
| 2 | `RmucSubRobotStatus` | `/robot_status` | `robot_status` | shared_ptr |
| 3 | `RmucSubRFIDStatus` | `/rfid_status` | `rfid_status` | — |
| 4 | `RmucSubRobotPosition` | `/robot_position` | `pose_x`, `pose_y`, `pose_yaw`, `is_at_nav_goal`, `pose` | — |
| 5 | `SubRadarTracks` | `/radar/enemy_tracks` | `radar_tracks` | — |
| 6 | `RmucSubSentryDecisionStatus` | `/sentry_decision_status` | `sentry_decision_status` | **NEW** |
| 7 | `RmucSubRobotBuff` | `/robot_buff` | `robot_buff` | **NEW** |
| 8 | `RmucSubProjectileAllowance` | `/projectile_allowance` | `projectile_allowance` | **NEW** |
| 9 | `RmucSubFieldStatus` | `/field_status` | `field_status` | **NEW** |
| 10 | `RmucSubEnemyMark` | `/enemy_mark` | `enemy_mark` | **NEW** |
| 11 | `RmucSubTeamPositions` | `/team_positions` | `team_positions` | **NEW** |
| 12 | `RmucSubTeamHP` | `/team_hp` | `team_hp` | **NEW** |

## 4. B. 初始化/配置插件（2 个）

| # | BT 节点名 | 类型 | 功能 | 关键端口 |
|:---:|:---|:---|:---|:---|
| 13 | `InitSentryConfig` | SyncAction | 初始化所有 `cfg.*` 配置参数 | hp_critical/low/safe, heat_high/critical, ammo_low/target, arrive_radius, objective_hold_ms, combat timings, coordinates, supply_zone_x/y (renamed from supply_x/y), supply_zone_x/y(**NEW**), heal_wait_ms/heal_min_ratio/search_timeout_ms(**NEW**) |
| 14 | `InitCmdState` | SyncAction | 初始化命令状态 | `cmd_state`(inout), `allow_ammo_target`(inout) |

## 5. C. 解析插件（1 个）

| # | BT 节点名 | 类型 | 功能 |
|:---:|:---|:---|:---|
| 15 | `ParseSentryBlackboard` | SyncAction | 集中解析器 |

- **输入**: 7 个原始话题 + 7 个新话题（共 14 路输入）
- **输出**: ~70+ 个黑板变量
- **关键输出**: `stage_remain_time`, `hp_cur/max`, `heat_cur`, `ammo_allow/left`, `base_hp_cur/max`, `outpost_alive`, `is_dead`, `is_weak`, `has_target`, `best_target`, `base_threat`, `team_coins` 等

## 6. D. 决策插件（4 个）

| # | BT 节点名 | 类型 | 功能 | 备注 |
|:---:|:---|:---|:---|:---|
| 18 | `DecideEconomyCmd` | SyncAction | 经济决策 | 增加 instant_respawn_cost 等 7 个新输入 |
| 19 | `DecideRespawnCmd` | SyncAction | 复活决策 | 增加 can_free/instant_respawn 等 |

### 决策策略

- **DecidePosture**: 基地威胁→防御; 低血/高热→防御; 有目标且>50%HP→进攻
- **DecideEconomyCmd**: 脱战+低血→远程回血; 弹低→远程补弹; 基地威胁→大能量机关
- **DecideRespawnCmd**: 死亡时始终确认普通复活；基地受威胁且金币≥300 或剩余<60s 时兑换立即复活

## 7. E. 命令复用插件（1 个）

| # | BT 节点名 | 类型 | 功能 |
|:---:|:---|:---|:---|
| 20 | `SentryCmdMux` | RosTopicPubNode (Action) | 命令复用器 5Hz → 0x0120 |

- **消息**: `RMUCSentryCmd` · 话题 `/sentry_cmd`
- **输入**: `posture`, `confirm_respawn`, `confirm_instant_respawn`, `allow_ammo_target`, `trigger_remote_ammo`, `trigger_remote_hp`, `enable_big_energy`, `cmd_state` (inout)

## 8. F. 导航插件（4 个）

| # | BT 节点名 | 类型 | 功能 |
|:---:|:---|:---|:---|
| 21 | `SendGoal` | StatefulAction | 发送导航目标 (Nav2 geometry_msgs::PoseStamped) |
| 22 | `CancelNavGoal` | Action | 取消当前导航目标 |
| 23 | `MoveAround` | Action | 在当前位置附近小范围移动 |
| 24 | `KeepRunning` | Action | 永远返回 RUNNING（保持子树活跃） |

## 9. G. 战斗插件（3 个）

| # | BT 节点名 | 类型 | 功能 | 备注 |
|:---:|:---|:---|:---|:---|
| 25 | `SelectBestTarget` | SyncAction | 选择最佳火力目标 | 增加 enemy_*_vuln 输入 |
| 26 | `AimAtTarget` | StatefulAction | 解析 "id:x:y" 目标 → 云台瞄准 (RUNNING→SUCCESS) | — |
| 27 | `FireBurst` | StatefulAction | 开火 burst_ms → 暂停 pause_ms → SUCCESS | — |

## 10. H. 机器人控制插件（1 个）

| # | BT 节点名 | 类型 | 功能 |
|:---:|:---|:---|:---|
| 28 | `RmucRobotControl` | RosTopicPubNode (Action) | 控制底盘/云台/开火 |

- **消息**: `RMUCRobotControl` · 话题 `/robot_control`
- **输入**: `stop_gimbal_scan`, `chassis_spin`, `fire_enable`

## 11. I. 导航选择插件（8 个）

| # | BT 节点名 | 类型 | 功能 | 备注 |
|:---:|:---|:---|:---|:---|
| 29 | `SelectObjective` | SyncAction | 目标选择 | 大幅增强：field_* 状态、fortress_ammo、全坐标 |
| 30 | `HoldObjective` | StatefulAction | 占据目标点 hold_ms | 基地威胁或有敌人时提前结束 |
| 31 | `WaypointPatrol` | SyncAction | 3 点循环巡逻 | 到达后切换下一个 |
| 32 | `SelectNearestDispelCard` | SyncAction | 选最近驱散卡 | — |
| 33 | `SelectNearestResupplyStation` | SyncAction | 选最近补给站 | — |
| 34 | `SelectSafeRetreatGoal` | SyncAction | 选安全撤退点 | — |
| 35 | `HoldAndHeal` | StatefulAction | 等待回血（hp ≥ hp_safe 结束） | — |
| 36 | `HoldForSupplyAmmoTick` | StatefulAction | 等待补弹达标 | — |

## 12. J. 恢复插件（NEW，4 个）

| # | BT 节点名 | 类型 | 功能 | 备注 |
|:---:|:---|:---|:---|:---|
| 37 | `RmucNavControlCmd` | RosTopicPubNode (Action) | 导航控制命令 | cmd_type: 1=nav, 3=stop |
| 38 | `RmucWaitAndHeal` | Action | 等待回血（带超时检测） | — |
| 39 | `InitSearchTimerIfNeeded` | SyncAction | 初始化搜索计时器 | — |
| 40 | `RmucMicroSearchSupplyCard` | Action | 微搜索补给卡 | — |

## 13. K. 条件插件（16 个）

| # | BT 节点名 | 功能 | SUCCESS 条件 | 备注 |
|:---:|:---|:---|:---|:---|
| 41 | `RmucIsGameTime` | 比赛时间检查 | game_progress=4 且在时间范围内 | — |
| 42 | `RmucIsDead` | 是否死亡 | is_dead=true | — |
| 43 | `IsWeakness` | 是否弱势 | shooter_power_output 断电检测，20帧防抖 | — |
| 44 | `RmucIsHPBelow` | HP 阈值检查 | hp_cur < hp_threshold | — |
| 45 | `IsAmmoBelow` | 弹药阈值检查 | ammo_allow < ammo_low | — |
| 46 | `IsCriticalState` | 危急状态检查 | HP < hp_critical **或** heat > heat_critical | — |
| 47 | `IsBaseThreatened` | 基地威胁检查 | base_threat=true **或** 基地血量 < 50% | 增加 outpost_alive 输入 |
| 48 | `HasValidTarget` | 有效目标检查 | has_target=true **且** best_target 非空 | — |
| 49 | `IsCombatAllowed` | 允许战斗检查 | 非虚弱 **且** 弹量>0 **且** 热量<上限 **且** HP>安全线 | 改用 robot_status |
| 50 | `IsFireWindowOk` | 开火窗口检查 | 非虚弱 **且** 弹量>0 **且** 热量<上限 | 增加 current_posture 等输入 |
| 51 | `IsZoneCardDetected` | 区域卡检测 | RFID 踩到指定区域 | — |
| 52 | `ShouldChassisSpin` | 是否旋转底盘 | 满足旋转条件 | **NEW** |
| 53 | `IsAnyDispelCardDetected` | 驱散卡检测 | rfid_supply **或** rfid_base **或** rfid_outpost | — |
| 54 | `IsAtGoal` | 到达目标检查 | ‖pose − goal‖ < arrive_radius | — |
| 55 | `RmucIsAtNavGoal` | 导航到达检查 | is_at_nav_goal=true | **NEW** |
| 56 | `RmucIsSupplyCardDetected` | 补给卡检测 | rfid_status 黑板 bool 读取 | **NEW** |

## 14. L. 装饰器（1 个）

| # | BT 节点名 | 类型 | 功能 |
|:---:|:---|:---|:---|
| 57 | `RateController` | DecoratorNode | 限制子节点 tick 频率 |

## 15. RMUC 拆分消息字段速查

> 原始 `RMUC.msg` 已拆分为 8 个独立消息，各位于 `rm_decision_interfaces/msg/RMUC/` 目录。

### 📥 RMUCGameStatus (`/game_status`, 1Hz)
```
uint8   game_progress          # 4=比赛中
uint16  stage_remain_time      # 剩余秒数
```

### 📥 RMUCRobotStatus (`/robot_status`, 10Hz)
```
uint16  current_hp / max_hp / shooter_heat / heat_limit / cooling_rate
uint16  ammo_allow / ammo_left
bool    is_dead / is_weak / is_disengaged
float32 disengage_cd_s
bool    can_remote_heal / can_remote_ammo
uint16  team_coins
bool    can_respawn
uint8   respawn_countdown_s
uint16  base_hp_cur / base_hp_max
bool    outpost_alive
```

### 📥 RMUCRFIDStatus (`/rfid_status`, 事件驱动)
```
bool    rfid_supply / rfid_base_buff / rfid_outpost_buff
bool    rfid_fortress_ally / rfid_fortress_enemy
bool    rfid_central_highland / rfid_ladder_highland
```

### 📥 RMUCRobotPosition (`/robot_position`, 50Hz)
```
float32 pose_x / pose_y / pose_yaw
bool    is_at_nav_goal
```

### 📥 RMUCEnemyTracks (`/radar/enemy_tracks`, 10-30Hz)
```
uint8     enemy_count
uint8[]   enemy_robot_id
float32[] enemy_x / enemy_y / enemy_confidence
```

### 📤 RMUCSentryCmd (`/sentry_cmd`, 2Hz)
```
uint8   cmd_posture                  # 1攻/2防/3移
bool    cmd_confirm_respawn / cmd_confirm_instant_respawn
uint16  cmd_allow_ammo_target
bool    cmd_trigger_remote_ammo / cmd_trigger_remote_hp
bool    cmd_enable_big_energy
```

### 📤 RMUCRobotControl (`/robot_control`, 10Hz)
```
bool    stop_gimbal_scan / chassis_spin / fire_enable
```

### 📤 RMUCNavControlCmd (`/nav_control_cmd`, 按需)
```
int32   cmd_type                     # 0=无操作 1=开始 2=终止 3=原地
bool    emergency_stop
```

## 16. 子树与插件引用矩阵

> 共 12 棵子树被主树引用（`DeathAndRespawn.xml` 未被引用，属于孤立文件）。

| 子树 | 使用的插件 |
|:---|:---|
| PerceptionAndBlackboard | RmucSubGameStatus, RmucSubRobotStatus, RmucSubRFIDStatus, RmucSubRobotPosition, SubRadarTracks, RmucSubSentryDecisionStatus, RmucSubRobotBuff, RmucSubProjectileAllowance, RmucSubFieldStatus, RmucSubEnemyMark, RmucSubTeamPositions, RmucSubTeamHP, ParseSentryBlackboard |
| InitOnce | InitSentryConfig, InitCmdState |
| RespawnRecovery | RmucIsDead, RmucRobotControl, RmucNavControlCmd, RmucWaitAndHeal, RmucIsSupplyCardDetected, RmucIsAtNavGoal, RmucMicroSearchSupplyCard, InitSearchTimerIfNeeded, SendGoal, KeepRunning |
| CriticalSurvival | IsCriticalState, SelectSafeRetreatGoal, RmucRobotControl, CancelNavGoal, SendGoal, KeepRunning, RateController |
| BaseDefense | IsBaseThreatened, RmucRobotControl, SendGoal, RateController |
| EngageCombat | HasValidTarget, IsCombatAllowed, RmucRobotControl |
| CombatLoop | SelectBestTarget, AimAtTarget, IsFireWindowOk, FireBurst, RmucRobotControl, KeepRunning |
| SustainAndEconomy | HealPlan + AmmoPlan 子树引用 |
| HealPlan | RmucIsHPBelow, IsZoneCardDetected, HoldAndHeal, RmucRobotControl, SendGoal, KeepRunning, RateController |
| AmmoPlan | IsAmmoBelow, IsZoneCardDetected, HoldForSupplyAmmoTick, SelectNearestResupplyStation, RmucRobotControl, SendGoal, KeepRunning, RateController |
| ObjectivePlanner | SelectObjective, IsAtGoal, HoldObjective, RmucRobotControl, SendGoal, KeepRunning, RateController |
| PatrolAndScan | WaypointPatrol, RmucRobotControl, ShouldChassisSpin, SendGoal, KeepRunning, RateController |

## 17. 文件结构

```
rm_behavior_tree/
├── include/rm_behavior_tree/plugins/rmuc_2026/
│   ├── action/
│   │   ├── aim_at_target.hpp
│   │   ├── decide_economy_cmd.hpp
│   │   ├── decide_posture.hpp
│   │   ├── decide_respawn_cmd.hpp
│   │   ├── fire_burst.hpp
│   │   ├── hold_and_heal.hpp
│   │   ├── hold_for_supply_ammo_tick.hpp
│   │   ├── hold_objective.hpp
│   │   ├── init_cmd_state.hpp
│   │   ├── init_sentry_config.hpp
│   │   ├── init_search_timer_if_needed.hpp
│   │   ├── micro_search_supply_card.hpp
│   │   ├── nav_control_cmd.hpp
│   │   ├── parse_sentry_blackboard.hpp
│   │   ├── posture_publishing.hpp
│   │   ├── robot_control.hpp
│   │   ├── select_best_target.hpp
│   │   ├── select_nearest_dispel_card.hpp
│   │   ├── select_nearest_resupply_station.hpp
│   │   ├── select_objective.hpp
│   │   ├── select_safe_retreat_goal.hpp
│   │   ├── sentry_cmd_mux.hpp
│   │   ├── sub_game_status.hpp
│   │   ├── sub_radar_tracks.hpp
│   │   ├── sub_rfid_status.hpp
│   │   ├── sub_robot_position.hpp
│   │   ├── sub_robot_status.hpp
│   │   ├── sub_sentry_decision_status.hpp    ← NEW
│   │   ├── sub_robot_buff.hpp                ← NEW
│   │   ├── sub_projectile_allowance.hpp      ← NEW
│   │   ├── sub_field_status.hpp              ← NEW
│   │   ├── sub_enemy_mark.hpp                ← NEW
│   │   ├── sub_team_positions.hpp            ← NEW
│   │   ├── sub_team_hp.hpp                   ← NEW
│   │   ├── wait_and_heal.hpp
│   │   └── waypoint_patrol.hpp
│   └── condition/
│       ├── has_valid_target.hpp
│       ├── is_ammo_below.hpp
│       ├── is_any_dispel_card_detected.hpp
│       ├── is_at_goal.hpp
│       ├── is_at_nav_goal.hpp
│       ├── is_base_threatened.hpp
│       ├── is_combat_allowed.hpp
│       ├── is_critical_state.hpp
│       ├── is_dead.hpp
│       ├── is_fire_window_ok.hpp
│       ├── is_game_time.hpp
│       ├── is_hp_below.hpp
│       ├── is_supply_card_detected.hpp
│       ├── is_weakness.hpp
│       ├── is_zone_card_detected.hpp
│       └── should_chassis_spin.hpp           ← NEW
├── plugins/rmuc_2026/  (同名 cpp 文件)
├── config/rmuc_2026/
│   ├── rmuc_2026.xml           (主树)
│   ├── PerceptionAndBlackboard.xml
│   ├── InitOnce.xml
│   ├── CommandHub.xml
│   ├── RespawnRecovery.xml
│   ├── CriticalSurvival.xml
│   ├── BaseDefense.xml
│   ├── EngageCombat.xml
│   ├── CombatLoop.xml
│   ├── SustainAndEconomy.xml
│   ├── HealPlan.xml
│   ├── AmmoPlan.xml
│   ├── ObjectivePlanner.xml
│   ├── PatrolAndScan.xml
│   ├── DeathAndRespawn.xml     (孤立文件，未被引用)
│   └── instruction/
│       └── rmuc_2026_plugins.ipynb  ← 本文档
└── CMakeLists.txt (rmuc_* 插件目标)
```

## 18. 入口 & 话题 & 构建

### 入口文件

RMUC 行为树使用独立入口 `rm_behavior_tree_rmuc.cpp`，与 RMUL 的 `rm_behavior_tree.cpp` 完全分离。

| 入口 | 可执行文件 | Groot2 端口 | 默认 XML |
|:---|:---|:---:|:---|
| RMUL | `rm_behavior_tree` | 1667 | `config/attack_left.xml` |
| **RMUC** | **`rm_behavior_tree_rmuc`** | **1668** | **`config/rmuc_2026/rmuc_2026.xml`** |

### 独立话题 (拆分后)

每个 RMUC 订阅/发布插件使用 **独立的 RosNodeParams 和话题**，不再共用 `/rmuc`：

| 方向 | BT 节点名 | 话题 | 消息类型 |
|:---:|:---|:---|:---|
| 📥 | `RmucSubGameStatus` | `/game_status` | `RMUCGameStatus` |
| 📥 | `RmucSubRobotStatus` | `/robot_status` | `RMUCRobotStatus` |
| 📥 | `RmucSubRFIDStatus` | `/rfid_status` | `RMUCRFIDStatus` |
| 📥 | `RmucSubRobotPosition` | `/robot_position` | `RMUCRobotPosition` |
| 📥 | `SubRadarTracks` | `/radar/enemy_tracks` | `RMUCEnemyTracks` |
| 📥 | `RmucSubSentryDecisionStatus` | `/sentry_decision_status` | NEW |
| 📥 | `RmucSubRobotBuff` | `/robot_buff` | NEW |
| 📥 | `RmucSubProjectileAllowance` | `/projectile_allowance` | NEW |
| 📥 | `RmucSubFieldStatus` | `/field_status` | NEW |
| 📥 | `RmucSubEnemyMark` | `/enemy_mark` | NEW |
| 📥 | `RmucSubTeamPositions` | `/team_positions` | NEW |
| 📥 | `RmucSubTeamHP` | `/team_hp` | NEW |
| 📤 | `SentryCmdMux` | `/sentry_cmd` | `RMUCSentryCmd` |
| 📤 | `RmucRobotControl` | `/robot_control` | `RMUCRobotControl` |
| 📤 | `RmucNavControlCmd` | `/nav_control_cmd` | `RMUCNavControlCmd` |
| 📤 | `SendGoal` | `navigate_to_pose` | PoseStamped (Nav2) |

### 构建与运行

```bash
# 编译（两个包）
cd ~/rm_code/2026_1_21/ros2_ws
colcon build --packages-select rm_decision_interfaces rm_behavior_tree

# 运行 RMUC 行为树
source install/setup.bash
ros2 run rm_behavior_tree rm_behavior_tree_rmuc \
  --ros-args -p style:=$(pwd)/src/rm_behavior_tree/rm_behavior_tree/config/rmuc_2026/rmuc_2026.xml

# 另一终端验证话题
ros2 topic list | grep -E "game_status|robot_status|rfid_status|robot_position|enemy_tracks|sentry_cmd|robot_control|nav_control|sentry_decision|robot_buff|projectile_allowance|field_status|enemy_mark|team_positions|team_hp"
```

## 19. 扩展指南

### 添加新插件
1. 在 `include/rm_behavior_tree/plugins/rmuc_2026/{action,condition}/` 创建 `.hpp`
2. 在 `plugins/rmuc_2026/{action,condition}/` 创建 `.cpp`
3. 在 `CMakeLists.txt` 添加 `ament_auto_add_library(rmuc_xxx SHARED ...)`
4. 如需新消息字段，在 `rm_decision_interfaces/msg/RMUC/` 目录下对应的 `.msg` 文件中添加
5. 在对应子树 XML 中引用，并在 `rmuc_2026.xml` 的 `TreeNodesModel` 中添加声明
6. 在 `rm_behavior_tree_rmuc.cpp` 中使用对应话题的 `RosNodeParams` 注册插件

### 添加新话题
若需新增独立话题（例如新的传感器输入）：
1. 在 `rm_decision_interfaces/msg/RMUC/` 下新建 `.msg`
2. 在 `rm_decision_interfaces/CMakeLists.txt` 中注册
3. 在 `rm_behavior_tree_rmuc.cpp` 中新建对应的 `BT::RosNodeParams`

### 标记说明
- 源码中带 `TODO:` 的地方表示需要根据实际硬件/赛事规则完善的逻辑
- `AimAtTarget` / `FireBurst` 需要与实际云台/射击控制器对接
- `ParseSentryBlackboard` 中的基地威胁判定需结合实际雷达数据细化